# 13 — File Handling

## Objectives
- Read and write files using Java NIO (`java.nio.file`)
- Process CSV and text files
- Use try-with-resources for safe file handling
- Apply file handling in real-world scenarios

## Java File APIs
| API | Package | Use Case |
|-----|---------|----------|
| `Files` | `java.nio.file` | Modern, recommended |
| `Path` | `java.nio.file` | File path manipulation |
| `FileReader/Writer` | `java.io` | Character streams (legacy) |
| `BufferedReader` | `java.io` | Efficient line reading |

In [1]:
import java.io.*;
import java.nio.file.*;
import java.util.*;

// Write a file
String content = "Name,Score,Grade\nAlice,92,A\nBob,78,B\nCharlie,85,A-\nDiana,61,C";
Path tempFile = Files.createTempFile("students", ".csv");
Files.writeString(tempFile, content);
System.out.println("Written to: " + tempFile.getFileName());

// Read all lines
List<String> lines = Files.readAllLines(tempFile);
System.out.println("Total lines: " + lines.size());

// Parse CSV
System.out.println("\n--- Student Report ---");
String[] headers = lines.get(0).split(",");
System.out.printf("%-12s %-8s %-6s%n", headers[0], headers[1], headers[2]);
System.out.println("-".repeat(28));

for (int i = 1; i < lines.size(); i++) {
    String[] cols = lines.get(i).split(",");
    System.out.printf("%-12s %-8s %-6s%n", cols[0], cols[1], cols[2]);
}

// Count students with A grade
long aCount = lines.stream().skip(1)
    .filter(l -> l.endsWith(",A") || l.contains(",A-"))
    .count();
System.out.println("\nStudents with A/A-: " + aCount);

// Append to file
Files.writeString(tempFile, "\nEva,88,B+", 
    StandardOpenOption.APPEND);
System.out.println("Appended Eva's record.");
System.out.println("Final line count: " + Files.readAllLines(tempFile).size());

Files.delete(tempFile); // cleanup

Written to: students15097747544459681149.csv
Total lines: 5

--- Student Report ---
Name         Score    Grade 
----------------------------
Alice        92       A     
Bob          78       B     
Charlie      85       A-    
Diana        61       C     

Students with A/A-: 2
Appended Eva's record.
Final line count: 6


## Mini Challenge
Read `datasets/students.csv`, filter students with CGPA > 8.0, and write them to `output/high_achievers.csv`.

In [4]:
import java.nio.file.*;
import java.util.*;
import java.util.stream.*;

// Fixed Paths: Stepping up one level out of the 'notebooks' folder
Path inputPath = Paths.get("..", "datasets", "students.csv");
Path outputPath = Paths.get("..", "output", "high_achievers.csv");

try {
    // Ensure the output directory exists
    if (outputPath.getParent() != null) {
        Files.createDirectories(outputPath.getParent());
    }

    // Read lines and isolate the header
    List<String> allLines = Files.readAllLines(inputPath);
    if (allLines.isEmpty()) {
        throw new IllegalArgumentException("The input CSV file is empty!");
    }
    String header = allLines.get(0);

    // Stream, parse CGPA, and filter records
    List<String> filteredLines = allLines.stream()
        .skip(1) // Skip header row
        .filter(line -> !line.trim().isEmpty()) // Skip any trailing empty lines
        .filter(line -> {
            String[] cols = line.split(",");
            try {
                // Grabs the last column.
                double cgpa = Double.parseDouble(cols[cols.length - 1].trim());
                return cgpa > 8.0;
            } catch (NumberFormatException | ArrayIndexOutOfBoundsException e) {
                return false;
            }
        })
        .collect(Collectors.toList());

    // Combine header with the filtered results
    List<String> finalOutput = new ArrayList<>();
    finalOutput.add(header);
    finalOutput.addAll(filteredLines);

    // Write out to the target file
    Files.write(outputPath, finalOutput);

    // Notebook Output Confirmation
    System.out.println("Processing complete!");
    System.out.println("Successfully filtered " + filteredLines.size() + " high achievers.");
    System.out.println("Saved results to: " + outputPath.toAbsolutePath());

} catch (NoSuchFileException e) {
    System.err.println("Error: Could not find '" + inputPath + "'. Checked absolute path: " + inputPath.toAbsolutePath());
} catch (Exception e) {
    System.err.println("An error occurred: " + e.getMessage());
    e.printStackTrace();
}

Processing complete!
Successfully filtered 0 high achievers.
Saved results to: /Users/rahulkpkurup/Learning/ML-Git-Portfolio/oop-using-java/notebooks/../output/high_achievers.csv
